In [1]:
import pandas as pd
import numpy as nps
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem import Descriptors
from tqdm import tqdm
import logging
from func_timeout import func_timeout, FunctionTimedOut
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import feyn
from mordred import Calculator, descriptors

logging.basicConfig(filename='descriptor_errors.log', level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

In [2]:
acidic_file = "./acidic_pKa_cleaned_final.csv"
smiles_col = 'SMILES'
target = 'pKa'

df = pd.read_csv(acidic_file)
df[target] = pd.to_numeric(df[target], errors='coerce')

In [3]:
saltRemover = SaltRemover(defnFilename='./Salts.txt')
df['Mol'] = df[smiles_col].astype(str).apply(
    lambda s: saltRemover.StripMol(Chem.MolFromSmiles(s))
)

In [4]:
calc = Calculator(descriptors, ignore_3D=True)

def safe_call(func, mol, timeout=1):
    try:
        return func_timeout(timeout, func, args=(mol,))
    except (FunctionTimedOut, Exception) as e:
        logging.info(f"Error in {func.__name__}: {e}")
        return np.nan

def compute_rdkit_descriptors(mol):
    descriptor_funcs = {name: func for name, func in Descriptors.descList}
    if mol is None or Chem.MolToSmiles(mol) == '':
        return None
    return {name: safe_call(func, mol, timeout=1) for name, func in descriptor_funcs.items()}

def compute_mordred_descriptors(mols):
    try:
        return calc.pandas(mols)
    except Exception as e:
        logging.info(f"Mordred batch error: {e}")
        return pd.DataFrame()

def compute_descriptors_for_df(df):
    mols = df['Mol'].tolist()
    rdkit_list = []
    for mol in tqdm(mols, desc="Computing RDKit descriptors"):
        rdkit_desc = compute_rdkit_descriptors(mol)
        rdkit_list.append(rdkit_desc if rdkit_desc is not None else {})
    rdkit_df = pd.DataFrame(rdkit_list)
    mordred_df = compute_mordred_descriptors(mols)
    full_desc_df = pd.concat([rdkit_df, mordred_df], axis=1)
    non_zero_std_cols = full_desc_df.std(numeric_only=True)
    full_desc_df = full_desc_df[non_zero_std_cols[non_zero_std_cols > 0].index]
    full_desc_df = full_desc_df.apply(pd.to_numeric, errors='coerce')
    full_df = pd.concat([df[[target]].reset_index(drop=True), full_desc_df.reset_index(drop=True)], axis=1)
    return full_df.dropna()

In [5]:
data = compute_descriptors_for_df(df)
train_data, test_data = train_test_split(data, test_size=0.25, random_state=42)

  2%|█▎                                                                            | 157/9191 [00:02<01:20, 111.56it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Autocorrelation.py:97: RuntimeWarning: Mean of empty slice.
  return avec - avec.mean()
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Constitutional.py:80: RuntimeWarning: invalid value encountered in double_scalars
  return S / self.mol.GetNumAtoms()


 17%|█████████████▌                                                                | 1592/9191 [00:20<01:49, 69.17it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 22%|█████████████████▌                                                            | 2066/9191 [00:27<01:27, 81.11it/s]

...

 66%|███████████████████████████████████████████████████▏                         | 6110/9191 [01:09<00:26, 115.33it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 78%|████████████████████████████████████████████████████████████▌                 | 7135/9191 [01:18<00:27, 74.63it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Autocorrelation.py:97: RuntimeWarning: Mean of empty slice.
  return avec - avec.mean()
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Constitutional.py:80: RuntimeWarning: invalid value encountered in double_scalars
  return S / self.mol.GetNumAtoms()
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Autocorrelation.py:97: RuntimeWarning: Mean of empty slice.
  return avec - avec.mean()
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Constitutional.py:80: RuntimeWarning: invalid value encoun

100%|██████████████████████████████████████████████████████████████████████████████| 9191/9191 [01:44<00:00, 88.16it/s]


In [8]:
# Clean up column issues before training
def sanitize_dataframe(df):
    df = df.loc[:, df.columns.notna()]  # Drop unnamed columns
    df = df[[col for col in df.columns if isinstance(col, str)]]  # Drop non-string columns
    df.columns = [str(col) for col in df.columns]  # Make sure all columns are strings
    return df

train_data = sanitize_dataframe(train_data)
test_data = sanitize_dataframe(test_data)

# Train QLattice
ql = feyn.QLattice()
models = ql.auto_run(train_data, output_name=target, n_epochs=200, threads=16, max_complexity=200, criterion='bic')

# Evaluate model
best_model = models[0]
y_true = test_data[target].values
y_pred = best_model.predict(test_data)
r2 = r2_score(y_true, y_pred)

print("\n--- QLattice Performance (RDKit + Mordred descriptors) ---")
print(f"Test R²: {r2:.3f}")
best_model.plot(train_data, test_data)
best_model.plot_regression(test_data)


AttributeError: 'DataFrame' object has no attribute 'name'

In [9]:
# Sanitize train/test data to remove weird columns
def fully_clean_dataframe(df):
    # Drop unnamed or null-named columns
    df = df.loc[:, df.columns.notna()]
    
    # Drop any column that is not a valid Series (like nested DF or broken object)
    valid_cols = []
    for col in df.columns:
        try:
            series = df[col]
            _ = series.name  # must have a name
            valid_cols.append(col)
        except Exception:
            print(f"Dropping invalid column: {col}")
    
    df = df[valid_cols]
    df.columns = [str(col) for col in df.columns]
    return df.reset_index(drop=True)

train_data = fully_clean_dataframe(train_data)
test_data = fully_clean_dataframe(test_data)


Dropping invalid column: BalabanJ
Dropping invalid column: BalabanJ
Dropping invalid column: BalabanJ
Dropping invalid column: BalabanJ
Dropping invalid column: BertzCT
Dropping invalid column: BertzCT
Dropping invalid column: BertzCT
Dropping invalid column: BertzCT
Dropping invalid column: BertzCT
Dropping invalid column: BertzCT
Dropping invalid column: BertzCT
Dropping invalid column: BertzCT
Dropping invalid column: LabuteASA
Dropping invalid column: LabuteASA
Dropping invalid column: LabuteASA
Dropping invalid column: LabuteASA
Dropping invalid column: LabuteASA
Dropping invalid column: LabuteASA
Dropping invalid column: LabuteASA
Dropping invalid column: LabuteASA
Dropping invalid column: PEOE_VSA1
Dropping invalid column: PEOE_VSA1
Dropping invalid column: PEOE_VSA1
Dropping invalid column: PEOE_VSA1
Dropping invalid column: PEOE_VSA1
Dropping invalid column: PEOE_VSA1
Dropping invalid column: PEOE_VSA1
Dropping invalid column: PEOE_VSA1
Dropping invalid column: PEOE_VSA10
Drop

In [10]:
ql = feyn.QLattice()
models = ql.auto_run(train_data, output_name=target, n_epochs=200, threads=16, max_complexity=200, criterion='bic')


[INFO: feyn.tools._data._types] - Column 'SsGeH3' is constant (contains only one value) and will be skipped.
[INFO: feyn.tools._data._types] - Column 'SsssAs' is constant (contains only one value) and will be skipped.
[INFO: feyn.tools._data._types] - Column 'SsssdAs' is constant (contains only one value) and will be skipped.
[INFO: feyn.tools._data._types] - Column 'SsSeH' is constant (contains only one value) and will be skipped.
[INFO: feyn.tools._data._types] - Column 'SdSe' is constant (contains only one value) and will be skipped.
[INFO: feyn.tools._data._types] - Column 'SssSe' is constant (contains only one value) and will be skipped.
[INFO: feyn.tools._data._types] - Column 'SaaSe' is constant (contains only one value) and will be skipped.
[INFO: feyn.tools._data._types] - Column 'SdssSe' is constant (contains only one value) and will be skipped.
